# CS781 Phase 2: ModernBERT-style encoder (written for architecture coverage)

> **STATUS: NOT RUN. This notebook has no leaderboard score and no results.** It applies the Phase 1 ModernBERT-style encoder to Phase 2, completing the architecture coverage of the repository. It is a new token-classification tagger (BIO tags over 512-token windows) followed by the same counsel / JUDGE rule layer as the submitted Phase 2 pipeline, with the Phase 1 backbone partially fine-tuned (embeddings frozen) under a time cap.
>
> **What was checked:** the code was run end to end on CPU with a tiny randomly initialised model and synthetic data (`SELFTEST = True` in the config cell) to verify that it runs and writes a schema-valid `prediction.json`; it was not run with the real checkpoint or the real data.
>
> **Attach:** Kaggle dataset `oipranesh/cs781-phase1-v3-encoder-modernbert` (`final_model.pt` + `legal_bpe_32k_mask.json`). **Secrets:** `HF_TOKEN` (the course data are gated). **Settings:** Internet ON, GPU T4 x1.


In [ ]:
import os
ROOT = os.environ.get('CS781_ROOT', '/kaggle/working'); os.makedirs(ROOT, exist_ok=True)
ARCH_LOADERS_SRC = '"""arch_loaders.py - load any of the four pre-trained architectures as a backbone for classification / tagging.\n   load_backbone(path) -> (backbone, hidden_size, causal, max_len, family)\n   backbone(input_ids, attention_mask) -> object with .last_hidden_state of shape [B, T, hidden_size]"""\nimport torch, torch.nn as nn\nfrom types import SimpleNamespace\nfrom transformers import GPT2Config, GPT2LMHeadModel, ModernBertConfig, ModernBertForMaskedLM, T5Config, T5ForConditionalGeneration\n\nclass BiLSTMLM(nn.Module):\n    def __init__(self, vocab, emb=512, hidden=1024, layers=3):\n        super().__init__()\n        self.vocab, self.emb_dim, self.hidden, self.layers = vocab, emb, hidden, layers\n        self.emb = nn.Embedding(vocab, emb); self.fwd = nn.LSTM(emb, hidden, layers, batch_first=True); self.bwd = nn.LSTM(emb, hidden, layers, batch_first=True)\n        self.norm_f, self.norm_b = nn.LayerNorm(hidden), nn.LayerNorm(hidden)\n        self.proj_f, self.proj_b = nn.Linear(hidden, emb), nn.Linear(hidden, emb); self.out_bias = nn.Parameter(torch.zeros(vocab))\n    def hidden_states(self, input_ids, attention_mask=None):\n        x = input_ids; B, T = x.shape\n        lengths = attention_mask.sum(1) if attention_mask is not None else torch.full((B,), T, device=x.device)\n        rev = (lengths[:, None] - 1 - torch.arange(T, device=x.device)[None, :]).clamp(min=0)\n        hf, _ = self.fwd(self.emb(x)); hb, _ = self.bwd(self.emb(x.gather(1, rev)))\n        hb = hb.gather(1, rev[:, :, None].expand(-1, -1, hb.shape[-1]))\n        return torch.cat([hf, hb], dim=-1)\n\nclass Backbone(nn.Module):\n    """one calling convention for every family (matches what the Phase 3 SeqClassifier expects)"""\n    def __init__(self, family, inner, embed):\n        super().__init__(); self.family, self.inner, self._embed = family, inner, embed\n    def get_input_embeddings(self): return self._embed\n    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None, use_cache=False, **kw):\n        f = self.family\n        if f == "gpt2": h = self.inner(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds, use_cache=False).last_hidden_state\n        else:\n            assert inputs_embeds is None, f"prompt tuning (inputs_embeds) is only wired for the gpt2 family, not {f}"\n            if f in ("modernbert", "t5"): h = self.inner(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state\n            elif f == "bilstm": h = self.inner.hidden_states(input_ids, attention_mask)\n            else: raise ValueError(f)\n        return SimpleNamespace(last_hidden_state=h)\n\nCAUSAL = {"gpt2": True, "modernbert": False, "t5": False, "bilstm": False}\n# modules to adapt with LoRA / LoRMA / IA3 (None = full fine-tuning only). ModernBERT\'s q,k,v are ONE fused linear, \'Wqkv\' (rows: q | k | v).\nADAPTER_TARGETS = {"gpt2": "attn.c_attn (fused Conv1D: q|k|v)", "modernbert": "attn.Wqkv (fused Linear: q|k|v)", "t5": "SelfAttention q, k, v, o (Linear)", "bilstm": None}\n\ndef build_full(mc):\n    f, V, d = mc["family"], mc["vocab_size"], mc["hidden_size"]\n    if f == "gpt2":\n        return GPT2LMHeadModel(GPT2Config(vocab_size=V, n_positions=mc["context_length"], n_ctx=mc["context_length"], n_embd=d, n_layer=mc["num_layers"], n_head=mc["num_heads"], n_inner=mc.get("n_inner", 4 * d),\n                                          resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0))\n    if f == "modernbert":\n        return ModernBertForMaskedLM(ModernBertConfig(vocab_size=V, max_position_embeddings=mc["context_length"], hidden_size=d, intermediate_size=mc["intermediate_size"], num_hidden_layers=mc["num_layers"],\n                                                      num_attention_heads=mc["num_heads"], local_attention=mc["local_attention"], global_attn_every_n_layers=mc["global_attn_every_n_layers"], pad_token_id=0,\n                                                      attention_dropout=0.0, mlp_dropout=0.0, embedding_dropout=0.0, classifier_pooling="mean", sparse_prediction=True, sparse_pred_ignore_index=-100, reference_compile=False))\n    if f == "t5":\n        return T5ForConditionalGeneration(T5Config(vocab_size=V, d_model=d, d_kv=mc["d_kv"], d_ff=mc["d_ff"], num_layers=mc["num_layers"], num_decoder_layers=mc["num_decoder_layers"], num_heads=mc["num_heads"],\n                                                   feed_forward_proj="gated-gelu", dropout_rate=0.0, tie_word_embeddings=True))\n    if f == "bilstm": return BiLSTMLM(V, emb=mc["emb"], hidden=mc["lstm_hidden"], layers=mc["num_layers"])\n    raise ValueError(f)\n\ndef load_backbone(path, device="cpu"):\n    ck = torch.load(path, map_location="cpu", weights_only=False); mc = ck["model_config"]; full = build_full(mc); full.load_state_dict(ck["model_state_dict"], strict=True)\n    f = mc["family"]\n    if f == "gpt2": inner, embed = full.transformer, full.transformer.wte\n    elif f == "modernbert": inner, embed = full.model, full.model.embeddings.tok_embeddings\n    elif f == "t5": inner, embed = full.encoder, full.shared\n    elif f == "bilstm": inner, embed = full, full.emb\n    else: raise ValueError(f)\n    return Backbone(f, inner, embed).to(device), mc["hidden_size"], CAUSAL[f], mc["context_length"], f\n'
open(ROOT + '/arch_loaders.py', 'w', encoding='utf-8').write(ARCH_LOADERS_SRC)   # loader for the four Phase 1 architectures
RULES_SRC = 'import re, random\n\nA_, R_ = \'A.COUNSEL\', \'R.COUNSEL\'\n\nchosen = \'R\'   # counsel default policy; select_defaults() re-chooses it on fold 1\n\nENT = ["APP","RESP","A.COUNSEL","R.COUNSEL","JUDGE","WIT","AUTH","COURT","STAT","PREC","DATE","CASENO"]\n\nTYPES = ["O"] + ENT\n\nWEAK = ["RESP", "A.COUNSEL", "R.COUNSEL", "WIT"]\n\nHON = re.compile(r"\\b(?:Mr|Ms|Mrs|Shri|Sri|Smt|Dr)\\b\\.?[ \\t]*(?=[A-Z])")\n\nHON_RE  = re.compile(r"\\b(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\b\\.?[ \\t]*(?=[A-Z])")\n\nNAME_TOK = re.compile(r"[A-Z][A-Za-z\\.\']*")\n\nWS = re.compile(r"[ \\t]+")\n\nSIDE_A = re.compile(r"petitioner|appellant|applicant|plaintiff|complainant|prosecution", re.I)\n\nSIDE_R = re.compile(r"respondent|defendant|accused", re.I)\n\ndef rule_spans(text):\n    out = []\n    for m in HON_RE.finditer(text):\n        pos = m.end()\n        if text[pos:pos + 7] == "Justice":          # judges\' honorific, not counsel\n            continue\n        p, name_end = pos, None\n        while True:\n            t = NAME_TOK.match(text, p)\n            if not t: break\n            name_end = t.end()\n            w = WS.match(text, name_end)\n            if not w or not text[w.end():w.end() + 1].isupper(): break\n            p = w.end()\n        if name_end is None: continue\n        e = name_end\n        while e > pos and text[e - 1] in ".,;:\'": e -= 1\n        if e > pos: out.append((pos, e))\n    return out\n\nHON_BEFORE = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\.?[ \\t]*$")\n\nPARTY_A = r"(?:petitioners?|appellants?|applicants?|plaintiffs?|complainants?|prosecution|revisionists?)"\n\nPARTY_R = r"(?:respondents?|defendants?|accused|non-\\s*applicants?|opposite\\s+part(?:y|ies))"\n\nFOR = r"(?:for|on\\s+behalf\\s+of|appearing\\s+for|representing)"\n\nMODS = r"(?:(?:learned|first|second|third|\\d+\\w*|all|other)\\s+)*"\n\nCUE_A = re.compile(rf"\\b{FOR}\\s+(?:the\\s+)?{MODS}{PARTY_A}", re.I)\n\nCUE_R = re.compile(rf"\\b{FOR}\\s+(?:the\\s+)?{MODS}{PARTY_R}", re.I)\n\nHDR_A = re.compile(rf"{PARTY_A}s?[\\s\\.\\-:]*(?:through|by)\\s*[:\\-]?\\s*$", re.I)\n\nHDR_R = re.compile(rf"{PARTY_R}s?[\\s\\.\\-:]*(?:through|by)\\s*[:\\-]?\\s*$", re.I)\n\nHON_TAIL = re.compile(r"(?:Mr|Ms|Mrs|Shri|Sri|Smt)\\.?[ \\t]*$")\n\ndef _cues(text, lo, hi):\n    out = [(m.start(), m.end(), A_) for m in CUE_A.finditer(text, lo, hi)]\n    out += [(m.start(), m.end(), R_) for m in CUE_R.finditer(text, lo, hi)]\n    return out\n\ndef assign_side(text, s, e, order, prev, W_L=300, W_R=200):\n    for step in order:\n        if step == "right":\n            c = _cues(text, e, min(len(text), e + W_R))\n            if c: return min(c)[2], "right"\n        elif step == "left":\n            c = _cues(text, max(0, s - W_L), s)\n            c = [x for x in c if x[1] <= s]\n            if c: return max(c, key=lambda x: x[1])[2], "left"\n        elif step == "hdr":\n            tail = HON_TAIL.sub("", text[max(0, s - 80):s])\n            if HDR_A.search(tail): return A_, "hdr"\n            if HDR_R.search(tail): return R_, "hdr"\n        elif step == "inherit":\n            if prev and s - prev[0] <= 200 and prev[2] != "default":\n                return prev[1], "inherit"\n    return R_, "default"\n\nVARIANTS = {"V1 left-for": ["left"], "V2 right-for": ["right"], "V3 right>left": ["right", "left"],\n            "V4 left>right": ["left", "right"], "V5 hdr>right>left>inh": ["hdr", "right", "left", "inherit"],\n            "V6 right>hdr>left>inh": ["right", "hdr", "left", "inherit"]}\n\nORDER = ["hdr", "right", "left", "inherit"]\n\ndef counsel_spans(text, policy):\n    out, prev = [], None\n    for s, e in sorted(rule_spans(text)):\n        lab, src = assign_side(text, s, e, ORDER, prev)\n        prev = (e, lab, src)\n        if src == "default":\n            if policy == "drop": continue\n            if policy == "A": lab = A_\n        out.append((s, e, lab))\n    return out\n\ndef counsel_f1(texts, gold, policy):\n    G = {A_: set(), R_: set()}; P = {A_: set(), R_: set()}\n    for d, ents in gold.items():\n        for x in ents:\n            if x[2] in G: G[x[2]].add((d, int(x[0]), int(x[1])))\n    for d, text in texts.items():\n        for s, e, lab in counsel_spans(text, policy): P[lab].add((d, s, e))\n    f = {}\n    for lab in (A_, R_):\n        tp = len(G[lab] & P[lab]); p = tp / max(len(P[lab]), 1); r = tp / max(len(G[lab]), 1)\n        f[lab] = 2 * p * r / (p + r) if p + r else 0.0\n    return f\n\nLOOK = ["APP", "RESP", "WIT", "JUDGE"]\n\nTOK = r"(?:[A-Z]\\.(?:\\s?[A-Z]\\.)*|[A-Z][a-z\']+|[A-Z]{2,})"\n\nTOK_RE = re.compile(TOK)\n\nNAME = rf"{TOK}(?:[ \\t]+{TOK}){{0,3}}"\n\nSTOP = {"JUDGMENT","JUDGEMENT","ORDER","BENCH","CORAM","BEFORE","PRESENT","HON\'BLE","MR","MRS","MS",\n        "JUSTICE","THE","AND","OF","DATE","NEW","DELHI","BY","DELIVERED"}\n\ndef back_name(text, end, max_toks=4):\n    ws = max(0, end - 80)\n    toks = [(m.start() + ws, m.end() + ws) for m in TOK_RE.finditer(text[ws:end])]\n    run = []\n    for a, b in reversed(toks):\n        nxt = run[0][0] if run else end\n        if text[b:nxt].strip(" \\t") != "": break\n        if text[a:b].upper().strip(".") in STOP: break\n        run.insert(0, (a, b))\n        if len(run) == max_toks: break\n    return (run[0][0], run[-1][1]) if run else None\n\ndef trim(text, s, e):\n    while s < e and (text[s].isspace() or text[s] in ".,;:"): s += 1\n    while e > s and (text[e-1].isspace() or text[e-1] in ",;:"): e -= 1\n    return (s, e) if e > s else None\n\nA_RE = re.compile(rf"\\bJustice[ \\t]+({NAME})")\n\nB_RE = re.compile(r",[ \\t]*J{1,2}\\b\\.?")\n\nC_RE = re.compile(r"\\bJJ?\\.?[ \\t]*\\(([^()\\n]{3,45})\\)")\n\nD_RE = re.compile(r"(?:BENCH\\s*:|\\(J\\)|\\(J\\.\\))\\s*([A-Z][A-Za-z\']+,[ \\t]*[A-Z](?:[ \\t]*\\.?[ \\t]*[A-Z])*\\.?)[ \\t]*\\(J\\)")\n\nCAP   = r"[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ\'’\\-]*"\n\nINIT  = r"[A-Z]\\.(?:[ \\t]*[A-Z]\\.)*"\n\nWORD  = rf"(?:{INIT}|{CAP})"\n\nNAME2 = rf"{WORD}(?:[ \\t]+{WORD}){{0,3}}"\n\nTOK_RE = re.compile(WORD)\n\nAJ_RE = re.compile(rf"(?i:\\bjustice)[ \\t]+({NAME2})")\n\nB_RE  = re.compile(r",[ \\t]*J{1,2}\\b\\.?")\n\nB2_RE = re.compile(r"[ \\t]J{1,2}\\.?(?=[ \\t]*(?:in\\b|speaking|writing|observed|held\\b|was\\b|said\\b|while\\b|who\\b|delivered|dissent|agreed|also\\b|noted|stated))")\n\nC_RE  = re.compile(r"\\bJJ?\\.?[ \\t]*\\(([^()\\n]{3,45})\\)")\n\nD_RE  = re.compile(r"(?:BENCH\\s*:|\\(J\\)|\\(J\\.\\))\\s*([A-Z][A-Za-z\']+,[ \\t]*[A-Z](?:[ \\t]*\\.?[ \\t]*[A-Z])*\\.?)[ \\t]*\\(J\\)")\n\nE_RE  = re.compile(r"\\bJJ?\\.?[ \\t]*\\[[ \\t]*([A-Za-zÀ-ÿ\\.\'’ \\t\\-]{3,45}?)[ \\t]*\\]")\n\nF_RE  = re.compile(r"BENCH[ \\t]*:[ \\t]*([^\\n]{3,120}?)(?=[ \\t]*(?:JUDG(?:E)?MENT|CITATION|ACT\\b|\\n|$))")\n\nNAMES = ["A Justice NAME", "B NAME, J.", "B2 NAME J (no comma)", "C J.(NAME)", "D BENCH: X, Y (J)", "E J [NAME]", "F BENCH: A & B"]\n\ndef judge_patterns2(text):\n    out = {n: set() for n in NAMES}\n    def add(n, s, e):\n        sp = trim(text, s, e)\n        if sp: out[n].add(sp)\n    for m in AJ_RE.finditer(text): add(NAMES[0], m.start(1), m.end(1))\n    for rx, n in [(B_RE, NAMES[1]), (B2_RE, NAMES[2])]:\n        for m in rx.finditer(text):\n            bn = back_name(text, m.start())\n            if bn: add(n, *bn)\n    for rx, n in [(C_RE, NAMES[3]), (E_RE, NAMES[5])]:\n        for m in rx.finditer(text):\n            inner = m.group(1).strip()\n            if inner and inner[0].isupper() and inner.upper() not in {"J", "JJ"} and re.fullmatch(r"[A-Za-zÀ-ÿ\\.\'’, \\-]+", inner):\n                add(n, m.start(1), m.end(1))\n    for m in D_RE.finditer(text): add(NAMES[4], m.start(1), m.end(1))\n    for m in F_RE.finditer(text):\n        seg, base, pos = m.group(1), m.start(1), 0\n        for part in re.split(r"[ \\t]*(?:&|\\bAND\\b)[ \\t]*", seg):\n            k = seg.find(part, pos)\n            if k < 0: continue\n            pos = k + len(part)\n            part2 = re.sub(r"[ \\t]*\\((?:C?J|J\\.?)\\)\\s*$", "", part)\n            sp = trim(text, base + k, base + k + len(part2))\n            if sp and re.fullmatch(NAME2, text[sp[0]:sp[1]]): out[NAMES[6]].add(sp)\n    return out\n\ndef resolve(spans):\n    out = []\n    for s, e in sorted(set(spans), key=lambda x: (x[0], -(x[1] - x[0]))):\n        if out and s < out[-1][1]: continue\n        out.append((s, e))\n    return out\n\ndef judge_spans_doc(text, keep):\n    allsp = set()\n    for n, sp in judge_patterns2(text).items():\n        if n in keep: allsp |= sp\n    return resolve(allsp)\n\ndef merge_all(texts, champ_filtered, keep):\n    out = {}\n    for d, text in texts.items():\n        rules = list(counsel_spans(text, chosen)) + [(s, e, "JUDGE") for s, e in judge_spans_doc(text, keep)]\n        kept = [k for k in champ_filtered[d] if k[2] not in (A_, R_, "JUDGE")]\n        kept = [k for k in kept if not any(k[0] < re_ and k[1] > rs for rs, re_, _ in rules)]\n        out[d] = sorted(set(kept + rules))\n    return out'
open(ROOT + '/p2_rules_core.py', 'w', encoding='utf-8').write(RULES_SRC)   # counsel / JUDGE rule layer
CAP_MIN = 60   # minutes of training (a time cap, as in the other notebooks)


In [ ]:
import os, sys, re, json, math, time, random, glob, gc
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

SELFTEST = False
FAMILY = "modernbert"
EPOCHS = 2
MAX_LEN, STRIDE_PRED, BATCH = 512, 256, 8
LR_BB, LR_HEAD = 5e-4, 1e-3
WORK = Path(os.environ.get("P2_WORK", "/kaggle/working/phase2_modernbert")); WORK.mkdir(parents=True, exist_ok=True)
ENT = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
BIO = ["O"] + [f"{p}-{l}" for l in ENT for p in "BI"]; B2I = {b: i for i, b in enumerate(BIO)}
IGN = -100
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); AMP = device.type == "cuda" and FAMILY != "t5"   # T5 overflows in fp16, so it trains in fp32
random.seed(42); np.random.seed(42); torch.manual_seed(42)
T0 = time.time()
print("device:", device, "| SELFTEST:", SELFTEST, "| training cap (min):", CAP_MIN)   # CAP_MIN comes from the runner (cell 0)

In [ ]:
def find(name, roots=("/kaggle/input", "/kaggle/working")):
    for r in roots:
        h = sorted(glob.glob(f"{r}/**/{name}", recursive=True))
        if h: return h[0]
    return None

if SELFTEST:
    SD = WORK / "synth"; SD.mkdir(exist_ok=True)
    first = ["Ram", "Sita", "Arjun", "Mohan", "Priya", "Kiran", "Dev", "Anil"]; last = ["Kumar", "Sharma", "Verma", "Singh", "Rao", "Das"]
    def mk_doc(k):
        a, b, j = random.choice(first), random.choice(last), random.choice(last)
        t = f"IN THE SUPREME COURT OF INDIA\nCivil Appeal No. {k} of 2001\nShri {a} {b}, Advocate for the appellant.\nJustice {first[k % 8]} {j} delivered the judgment on 12.03.2001.\nMr. {b} {a}, Advocate, for the respondents.\n"
        g = []
        for m in re.finditer(r"SUPREME COURT OF INDIA", t): g.append([m.start(), m.end(), "COURT"])
        for m in re.finditer(r"\d\d\.\d\d\.\d{4}", t): g.append([m.start(), m.end(), "DATE"])
        for m in re.finditer(r"Justice (\w+ \w+)", t): g.append([m.start(1), m.end(1), "JUDGE"])
        m = re.search(r"Shri (\w+ \w+)", t); g.append([m.start(1), m.end(1), "A.COUNSEL"])
        m = re.search(r"Mr\. (\w+ \w+)", t); g.append([m.start(1), m.end(1), "R.COUNSEL"])
        return t, sorted(g)
    for fold, n in (("fold1", 40), ("fold2", 12), ("fold3", 8)):
        docs = [mk_doc(i + (0 if fold == "fold1" else 100)) for i in range(n)]
        json.dump({f"d{i}": d[0] for i, d in enumerate(docs)}, open(SD / f"{fold}-text.json", "w"))
        if fold != "fold3": json.dump({f"d{i}": d[1] for i, d in enumerate(docs)}, open(SD / f"{fold}-gold.json", "w"))
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    tk = Tokenizer(models.BPE(unk_token="[UNK]")); tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.train_from_iterator([json.load(open(SD / "fold1-text.json"))[k] for k in json.load(open(SD / "fold1-text.json"))], trainers.BpeTrainer(vocab_size=300, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"], initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
    tk.save(str(SD / "legal_bpe_32k_mask.json")); V = tk.get_vocab_size()
    sys.path.insert(0, ROOT); import arch_loaders as AL
    mc = dict(family="modernbert", vocab_size=V, hidden_size=64, intermediate_size=128, context_length=512, num_layers=2, num_heads=2, local_attention=16, global_attn_every_n_layers=2)
    torch.save(dict(model_config=mc, model_state_dict=AL.build_full(mc).state_dict()), SD / "final_model.pt")
    DATA_DIR, TOK_PATH, CKPT_PATH = SD, str(SD / "legal_bpe_32k_mask.json"), str(SD / "final_model.pt")
else:
    need = ["fold1-text.json", "fold1-gold.json", "fold2-text.json", "fold2-gold.json", "fold3-text.json"]
    if not all(find(n) for n in need):
        from huggingface_hub import snapshot_download
        tok = os.environ.get("HF_TOKEN")
        if not tok:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("HF_TOKEN")
        snapshot_download("Exploration-Lab/CS-781-Capstone", repo_type="dataset", allow_patterns=["Phase_2_LNER/*"], token=tok, local_dir=str(WORK / "hf"))
    DATA_DIR = Path(find("fold1-text.json", (str(WORK / "hf"), "/kaggle/input")) or find("fold1-text.json")).parent
    TOK_PATH = find("legal_bpe_32k_mask.json"); CKPT_PATH = None
    for p_ in sorted(glob.glob("/kaggle/input/**/final_model.pt", recursive=True)):      # the checkpoint whose model_config family is modernbert
        try: fam_ = torch.load(p_, map_location="cpu", weights_only=False)["model_config"].get("family", "gpt2")
        except Exception: continue
        if fam_ == FAMILY: CKPT_PATH = p_; break

    assert TOK_PATH and CKPT_PATH, "attach the dataset oipranesh/cs781-phase1-v3-encoder-modernbert (final_model.pt + legal_bpe_32k_mask.json)"
    sys.path.insert(0, ROOT); import arch_loaders as AL
def jl(n):   # fold1/fold2 live in .../Train, fold3 in the sibling .../Test, so search the parent folder too
    p = find(n, (str(DATA_DIR), str(Path(DATA_DIR).parent))); assert p, f"{n} not found under {DATA_DIR} or its parent"
    return json.load(open(p, encoding="utf-8"))
train_texts, train_gold, val_texts, val_gold, test_texts = jl("fold1-text.json"), jl("fold1-gold.json"), jl("fold2-text.json"), jl("fold2-gold.json"), jl("fold3-text.json")
print(f"fold1 {len(train_texts)} docs | fold2 {len(val_texts)} | fold3 {len(test_texts)} | tokenizer {TOK_PATH} | checkpoint {CKPT_PATH}")
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(TOK_PATH)

In [ ]:
def doc_tokens(text):
    e = TOK.encode(text, add_special_tokens=False); return e.ids, e.offsets
def bio_labels(offsets, gold):
    spans = sorted((int(x[0]), int(x[1]), x[2]) for x in gold); out = []; j = 0; last = None
    for a, b in offsets:
        while j < len(spans) and spans[j][1] <= a: j += 1
        if b > a and j < len(spans) and spans[j][0] < b and spans[j][1] > a:
            out.append(B2I[("I-" if last == j else "B-") + spans[j][2]]); last = j
        else: out.append(0); last = None if b > a else last
    return out
def make_windows(texts, golds):
    W = []
    for d, t in texts.items():
        ids, off = doc_tokens(t); lab = bio_labels(off, golds.get(d, []))
        for s in range(0, len(ids), MAX_LEN): W.append((ids[s:s + MAX_LEN], lab[s:s + MAX_LEN]))
    return W
TRW = make_windows(train_texts, train_gold); print("train windows:", len(TRW), "| tokens:", sum(len(w[0]) for w in TRW))
cnt = Counter(l for _, lab in TRW for l in lab); freq = torch.tensor([max(cnt.get(i, 0), 1) for i in range(len(BIO))], dtype=torch.float)
CLS_W = (freq[0] / freq).sqrt().clamp(1.0, 10.0); CLS_W[0] = 1.0

In [ ]:
bb, hid, causal, ctx, fam = AL.load_backbone(CKPT_PATH, device="cpu"); assert fam == FAMILY, (fam, FAMILY)
with torch.no_grad(): DIM = bb(input_ids=torch.zeros(1, 4, dtype=torch.long), attention_mask=torch.ones(1, 4, dtype=torch.long)).last_hidden_state.shape[-1]
class Tagger(nn.Module):
    def __init__(s, bb, dim, n): super().__init__(); s.bb = bb; s.drop = nn.Dropout(0.1); s.head = nn.Linear(dim, n)
    def forward(s, ids, mask): return s.head(s.drop(s.bb(input_ids=ids, attention_mask=mask).last_hidden_state))
model = Tagger(bb, DIM, len(BIO)).to(device)
for n_, p in model.bb.inner.named_parameters():
    if "embed" in n_.lower() or n_.startswith("shared") or n_.endswith("wte.weight") or n_.endswith("wpe.weight"): p.requires_grad = False   # partial fine-tuning: the embeddings stay frozen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad); print(f"modernbert tagger: feature dim {DIM} | trainable {trainable/1e6:.2f}M of {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
def collate(items):
    L = max(len(i) for i, _ in items); ids = torch.zeros(len(items), L, dtype=torch.long); m = torch.zeros_like(ids); y = torch.full_like(ids, IGN)
    for k, (i, l) in enumerate(items): ids[k, :len(i)] = torch.tensor(i); m[k, :len(i)] = 1; y[k, :len(l)] = torch.tensor(l)
    return ids.to(device), m.to(device), y.to(device)
opt = torch.optim.AdamW([{"params": [p for p in model.bb.parameters() if p.requires_grad], "lr": LR_BB}, {"params": model.head.parameters(), "lr": LR_HEAD}], weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda", enabled=AMP); lossf = nn.CrossEntropyLoss(weight=CLS_W.to(device), ignore_index=IGN)
order_len = sorted(range(len(TRW)), key=lambda i: len(TRW[i][0]))
batches = [order_len[i:i + BATCH] for i in range(0, len(order_len), BATCH)]; total = EPOCHS * len(batches); step = 0; t_train = time.time(); stop = False
for ep in range(EPOCHS):
    random.shuffle(batches); model.train(); run = 0.0; nb = 0
    for bi in batches:
        ids, m, y = collate([TRW[i] for i in bi])
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
            lo = model(ids, m); loss = lossf(lo.float().reshape(-1, len(BIO)), y.reshape(-1))
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update()
        run += loss.item(); nb += 1; step += 1
        if step % 50 == 0: print(f"  ep {ep + 1} step {step}/{total} loss {run / nb:.4f} | {(time.time() - t_train) / 60:.1f} min")
        if (time.time() - t_train) / 60 >= CAP_MIN: stop = True; break
    print(f"epoch {ep + 1} done: loss {run / max(nb, 1):.4f}")
    if stop: print(f"STOPPED by the {CAP_MIN}-minute cap after {step} steps"); break
print(f"training took {(time.time() - t_train) / 60:.1f} min")

In [ ]:
@torch.inference_mode()
def doc_probs(text):
    ids, off = doc_tokens(text)
    if not ids: return off, None
    P = np.zeros((len(ids), len(BIO)), dtype=np.float32); C = np.zeros(len(ids), dtype=np.float32); model.eval()
    for s in range(0, len(ids), STRIDE_PRED):
        e = min(s + MAX_LEN, len(ids)); x = torch.tensor([ids[s:e]], device=device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP): p = model(x, torch.ones_like(x)).float().softmax(-1)[0].cpu().numpy()
        P[s:e] += p; C[s:e] += 1
        if e == len(ids): break
    return off, P / np.maximum(C, 1)[:, None]
def decode(text, off, P):
    pred = P.argmax(-1); spans = []; cur = None
    def close():
        nonlocal cur
        if cur:
            s, e, lab, ps = cur;
            while s < e and text[s].isspace(): s += 1
            while e > s and text[e - 1].isspace(): e -= 1
            if s < e: spans.append((s, e, lab, float(np.mean(ps))))
        cur = None
    for (a, b), k, p in zip(off, pred, P):
        if b <= a: continue
        tag = BIO[int(k)]
        if tag == "O": close(); continue
        pre, lab = tag.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab: close(); cur = [a, b, lab, [p[k]]]
        else: cur[1] = b; cur[3].append(p[k])
    close(); return [tuple(x) for x in spans]
def predict(texts):
    out = {}
    for d, t in texts.items():
        off, P = doc_probs(t); out[d] = [] if P is None else decode(t, off, P)
    return out
def strict_macro(pred, gold):
    f = {}
    for lab in ENT:
        G = {(d, int(x[0]), int(x[1])) for d, e in gold.items() for x in e if x[2] == lab}
        if not G: continue
        Pp = {(d, s, e) for d, sp in pred.items() for (s, e, l, *_) in sp if l == lab}; tp = len(G & Pp)
        p = tp / len(Pp) if Pp else 0.0; r = tp / len(G); f[lab] = 2 * p * r / (p + r) if p + r else 0.0
    return float(np.mean(list(f.values()))) if f else 0.0, f
RULES = {"__name__": "p2_rules"}; exec(open(os.environ.get("P2_RULES_PATH", "/kaggle/working/p2_rules_core.py"), encoding="utf-8").read(), RULES)
def select_defaults(texts, gold):
    f1s = {pol: RULES["counsel_f1"](texts, gold, pol) for pol in ("R", "A", "drop")}
    RULES["chosen"] = max(f1s, key=lambda p: (f1s[p]["A.COUNSEL"] + f1s[p]["R.COUNSEL"], p == "R"))
    G = {(d, int(x[0]), int(x[1])) for d, e in gold.items() for x in e if x[2] == "JUDGE"}; Pn = {n: set() for n in RULES["NAMES"]}
    for d, t in texts.items():
        for n, sp in RULES["judge_patterns2"](t).items(): Pn[n] |= {(d, s, e) for s, e in sp}
    keep = [n for n in RULES["NAMES"] if len(G & Pn[n]) >= 5 and len(G & Pn[n]) / max(len(Pn[n]), 1) >= 0.4]
    print(f"rule layer chosen on fold 1 only: counsel default '{RULES['chosen']}' | JUDGE patterns kept: {keep}"); return keep
KEEP = select_defaults(train_texts, train_gold)
VAL_PRED = predict(val_texts); m0, f0 = strict_macro(VAL_PRED, val_gold)
VAL_RULES = RULES["merge_all"](val_texts, VAL_PRED, KEEP); m1, f1_ = strict_macro({d: [(s, e, l) for s, e, l, *_ in sp] for d, sp in VAL_RULES.items()}, val_gold)
print(f"FOLD 2 strict macro-F1 (12-label convention, labels present in gold): modernbert alone {m0:.4f} | modernbert + counsel/JUDGE rules {m1:.4f}")
print({k: round(v, 3) for k, v in f1_.items()})

In [ ]:
TEST_PRED = predict(test_texts); TEST_RULES = RULES["merge_all"](test_texts, TEST_PRED, KEEP)
def to_obj(P): return {str(d): [[int(s), int(e), l] for s, e, l, *_ in sp] for d, sp in P.items()}
def validate(obj, texts):
    bad = 0
    for d, sp in obj.items():
        t = texts[d]
        for s, e, l in sp:
            if not (0 <= s < e <= len(t)) or l not in ENT: bad += 1
    assert set(obj) == set(map(str, texts)), "document ids differ"; return bad
obj_rules, obj_model = to_obj({d: [(s, e, l) for s, e, l, *_ in sp] for d, sp in TEST_RULES.items()}), to_obj(TEST_PRED)
print("invalid spans: with rules", validate(obj_rules, {str(k): v for k, v in test_texts.items()}), "| model only", validate(obj_model, {str(k): v for k, v in test_texts.items()}))
json.dump(obj_rules, open(WORK / "prediction.json", "w", encoding="utf-8")); json.dump(obj_model, open(WORK / "prediction_model_only.json", "w", encoding="utf-8"))
json.dump(dict(fold2_modernbert_only=m0, fold2_modernbert_plus_rules=m1, per_label_with_rules=f1_, train_steps=step, minutes=round((time.time() - T0) / 60, 1)), open(WORK / "results.json", "w"), indent=1)
print("wrote", WORK / "prediction.json", "| total minutes", round((time.time() - T0) / 60, 1))